# 01 — Prueba de fuentes de datos (IEC15)

**Objetivo:** comprobar si yfinance entrega precios históricos de acciones chilenas desde 2019 hasta hoy, antes de calcular cualquier cosa del índice.

**Qué vamos a revisar:**
1. Si cada acción descarga datos o viene vacía.
2. Desde qué fecha y hasta qué fecha llegan los datos.
3. Si hay datos después del 17-jul-2026 (fecha en que se rompió el histórico de yfinance para Chile).
4. Si el IPSA (`^IPSA`) está disponible como benchmark.

**Importante:** los datos se guardan en `data/raw/`, que está excluida del repositorio. Antes de hacer commit de este notebook, usa **Edit → Clear Outputs of All Cells** para no publicar datos de Yahoo Finance.

In [ ]:
from pathlib import Path
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

print("yfinance versión:", yf.__version__)
print("pandas versión:", pd.__version__)

# --- Configuración ---
FECHA_INICIO = "2019-01-01"      # 6 meses antes de la fecha base (31-dic-2019) + margen
FECHA_QUIEBRE = "2026-07-17"     # desde aquí se rompió el histórico de yfinance para Chile

# Muestra de prueba: acciones grandes y conocidas de la Bolsa de Santiago + el IPSA
TICKERS = [
    "SQM-B.SN",       # SQM serie B
    "CHILE.SN",       # Banco de Chile
    "BSANTANDER.SN",  # Banco Santander Chile
    "FALABELLA.SN",   # Falabella
    "CENCOSUD.SN",    # Cencosud
    "COPEC.SN",       # Empresas Copec
    "CMPC.SN",        # CMPC
    "LTM.SN",         # LATAM Airlines (útil para probar eventos corporativos)
    "^IPSA",          # Benchmark
]

# Carpeta de datos: funciona si el notebook está en notebooks/ o en la raíz del repo
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CARPETA_RAW = RAIZ / "data" / "raw"
CARPETA_RAW.mkdir(parents=True, exist_ok=True)
print("Los datos se guardarán en:", CARPETA_RAW)

## Prueba 1: descargar cada ticker

`auto_adjust=False` entrega dos columnas de cierre:
- **Close**: ajustado solo por splits. Es el que usa un índice *Price Return* (v0.1).
- **Adj Close**: ajustado por splits y dividendos. Servirá para la versión *Total Return* (v0.2).

`actions=True` agrega las columnas de dividendos y splits, que necesitamos para los eventos corporativos.

In [ ]:
datos = {}
diagnostico = []

for ticker in TICKERS:
    try:
        df = yf.Ticker(ticker).history(start=FECHA_INICIO, auto_adjust=False, actions=True)
    except Exception as e:
        df = pd.DataFrame()
        print(f"{ticker}: error -> {e}")

    if df.empty:
        diagnostico.append({"ticker": ticker, "estado": "SIN DATOS"})
        continue

    df.index = df.index.tz_localize(None)   # quitar zona horaria para comparar fechas
    datos[ticker] = df

    diagnostico.append({
        "ticker": ticker,
        "estado": "OK",
        "filas": len(df),
        "primera_fecha": df.index.min().date(),
        "ultima_fecha": df.index.max().date(),
        "filas_post_quiebre": int((df.index > FECHA_QUIEBRE).sum()),
        "dias_volumen_cero": int((df["Volume"] == 0).sum()),
        "precios_faltantes": int(df["Close"].isna().sum()),
        "n_dividendos": int((df["Dividends"] > 0).sum()),
        "n_splits": int((df["Stock Splits"] > 0).sum()),
    })

tabla = pd.DataFrame(diagnostico)
tabla

## Cómo leer la tabla

- **estado = SIN DATOS** → yfinance no entrega nada para ese ticker.
- **ultima_fecha** cerca del 17-jul-2026 y **filas_post_quiebre = 0** → el problema del histórico nos afecta.
- **filas** esperadas: unas 245 por año hábil, o sea ~1.900 desde 2019. Muchas menos indican huecos.
- **dias_volumen_cero** alto → posibles días sin datos reales, o una acción poco líquida.

**Envíame una captura de esta tabla** para decidir la fuente de datos.

## Prueba 2: gráfico de control

Precios normalizados a 100 al inicio. Sirve para detectar a simple vista saltos raros (un split mal ajustado se ve como una caída vertical).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for ticker, df in datos.items():
    serie = df["Close"].dropna()
    ax.plot(serie.index, serie / serie.iloc[0] * 100, label=ticker, linewidth=1)

ax.axvline(pd.Timestamp("2019-12-31"), color="gray", linestyle="--", linewidth=1)
ax.axvline(pd.Timestamp(FECHA_QUIEBRE), color="red", linestyle="--", linewidth=1)
ax.set_title("Precios de cierre normalizados (base 100) — control de calidad")
ax.set_ylabel("Base 100")
ax.legend(fontsize=8, ncol=3)
plt.tight_layout()
plt.show()

## Guardar los datos descargados

Se guardan como CSV en `data/raw/` (excluida del repositorio).

In [ ]:
for ticker, df in datos.items():
    nombre = ticker.replace("^", "").replace(".", "_") + ".csv"
    df.to_csv(CARPETA_RAW / nombre)
    print("Guardado:", nombre)

In [ ]:
post = {t: df[df.index > FECHA_QUIEBRE] for t, df in datos.items()}
pd.DataFrame({
    t: {"filas": len(d),
        "dias_con_volumen": int((d["Volume"] > 0).sum()),
        "dias_precio_repetido": int((d["Close"].diff() == 0).sum()),
        "ultimo_dia_con_volumen": d.index[d["Volume"] > 0].max()}
    for t, d in post.items()
}).T